# Initialization

In [261]:
# Initialization
#import pyspark as spark
from pyspark.sql import SparkSession, functions

# incase the spark context already exists

spark = SparkSession \
        .builder \
        .appName("Project2") \
        .getOrCreate()


In [262]:
# Global Constants
random_seed = 42

path_to_file_local = "games_cleaned.csv"
path_to_file_hadoop = ""

In [263]:
from pyspark.sql.types import *


# TODO: Finish this... I'm too tired to finish this tonight
types = StructType([
    StructField('_c0', IntegerType(), True),
    StructField('AppId', IntegerType(), True),
    StructField('Name', StringType(), True),
    StructField('Release date', StringType(), True),
    StructField('Estimated owners', StringType(), True),
    StructField('Peak CCU', IntegerType(), True),
    StructField('Required age', IntegerType(), True),
    StructField('Price', DoubleType(), True),
    StructField('DiscountDLC count', IntegerType(), True),
    StructField('Supported languages', StringType(), True),
    StructField('Full audio languages', StringType(), True),
    StructField('Windows', IntegerType(), True),
    StructField('Mac', IntegerType(), True),
    StructField('Linux', IntegerType(), True),
    StructField('Metacritic score', DoubleType(), True),
    StructField('User score', DoubleType(), True),
    StructField('Positive', IntegerType(), True),
    StructField('Negative', IntegerType(), True),
    StructField('Average playtime forever', DoubleType(), True),
    StructField('Average playtime two weeks', DoubleType(), True),
    StructField('Median playtime forever', DoubleType(), True),
    StructField('Median playtime two weeks', DoubleType(), True),
    StructField('Developers', StringType(), True),
    StructField('Publishers', StringType(), True), 
    StructField('Categories', StringType(), True),
    StructField('Genres', StringType(), True),
    StructField('Tags', StringType(), True),
    StructField('estimated_owners_mid', IntegerType(), True),
    StructField('estimated_revenue', DoubleType(), True)
])


data = spark.read.schema(types).csv(path_to_file_local, header=True)
data.show()

+---+-------+----------------------------+------------+----------------+--------+------------+-----+-----------------+--------------------+--------------------+-------+----+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+
|_c0|  AppId|                        Name|Release date|Estimated owners|Peak CCU|Required age|Price|DiscountDLC count| Supported languages|Full audio languages|Windows| Mac|Linux|Metacritic score|User score|Positive|Negative|Average playtime forever|Average playtime two weeks|Median playtime forever|Median playtime two weeks|          Developers|          Publishers|          Categories|              Genres|                Tags|estimated_owners_mid|estimated_revenue|
+---+-------+----------------------------+------------+-

# Data Preparation

### Train Test Split

In [264]:
from pyspark.ml.feature import VectorAssembler

# Note: We probably need to translate some of the strings to integer enumerations...
# but until then....

string_columns = [c for c, t in data.dtypes if t == 'string']
#string_columns = [c.name for c in types.fields if c.typeName == 'str']
svm_data = data.drop(*string_columns)

assembler = VectorAssembler(
    inputCols=svm_data.columns,
    outputCol='features',
    handleInvalid='keep'
)

data_separeted = assembler.transform(svm_data.fillna(0) \
.withColumn('sold_well', functions.when(functions.col('estimated_revenue') >= 500000, 1).otherwise(0))
                                     ).drop('estimated_revenue')
data_separeted.show()

+---+-------+--------+------------+-----+-----------------+-------+---+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+---------+--------------------+
|_c0|  AppId|Peak CCU|Required age|Price|DiscountDLC count|Windows|Mac|Linux|Metacritic score|User score|Positive|Negative|Average playtime forever|Average playtime two weeks|Median playtime forever|Median playtime two weeks|estimated_owners_mid|sold_well|            features|
+---+-------+--------+------------+-----+-----------------+-------+---+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+---------+--------------------+
|  0|2539430|       0|           0|  0.0|                0|      0|  0|    0|             0.0|       0.0|       0|       0|                     0.0|                  

In [265]:
train, test = data_separeted.randomSplit([0.7, 0.3], seed=random_seed)
train.show()

+---+-------+--------+------------+-----+-----------------+-------+---+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+---------+--------------------+
|_c0|  AppId|Peak CCU|Required age|Price|DiscountDLC count|Windows|Mac|Linux|Metacritic score|User score|Positive|Negative|Average playtime forever|Average playtime two weeks|Median playtime forever|Median playtime two weeks|estimated_owners_mid|sold_well|            features|
+---+-------+--------+------------+-----+-----------------+-------+---+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+---------+--------------------+
|  0|2539430|       0|           0|  0.0|                0|      0|  0|    0|             0.0|       0.0|       0|       0|                     0.0|                  

In [266]:
# SVM for determining if a game will sell well
from pyspark.ml.classification import LinearSVC
svm = LinearSVC(labelCol='sold_well', featuresCol='features')

# default training
model =svm.fit(train)


In [267]:
prediction = model.transform(test)
prediction.show()


+---+-------+--------+------------+-----+-----------------+-------+---+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+---------+--------------------+--------------------+----------+
|_c0|  AppId|Peak CCU|Required age|Price|DiscountDLC count|Windows|Mac|Linux|Metacritic score|User score|Positive|Negative|Average playtime forever|Average playtime two weeks|Median playtime forever|Median playtime two weeks|estimated_owners_mid|sold_well|            features|       rawPrediction|prediction|
+---+-------+--------+------------+-----+-----------------+-------+---+-----+----------------+----------+--------+--------+------------------------+--------------------------+-----------------------+-------------------------+--------------------+---------+--------------------+--------------------+----------+
|  2|1034400|       0|           0| 4.99|                0|      0|  0

In [268]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

bcv = BinaryClassificationEvaluator(rawPredictionCol='rawPrediction', metricName='areaUnderROC')
auc = bcv.evaluate(prediction.withColumnRenamed("sold_well", "label"))
print(auc)


0.9999929261620962


In [269]:
spark.stop()